<a href="https://colab.research.google.com/github/sayu0303/CONTROLE-2025/blob/main/CONTROLE_2910.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***CONTROLE 2025***

*DATA: 29/10*

---

***Insatalar o pacote do python***

In [4]:
! pip install -qq control

In [5]:
#Definição das bibliotecas utilizadas

import numpy as np
from scipy import linalg
import control as ct
from control import linearize

#Definição da função de modelo a ser utilizado


def reator_batelada(t,x,u, params={}):

  #Declaração dos parametros

  umax = 0.2
  Ks = 1.0
  Yxs = 0.5
  Ypx = 0.2

  #Declaração das condições de alimentação (variaveis de entrada)

  Sf = u[0]
  F = u[1]

  #Declaração das condições iniciais (variaveis de estado) /// LEMBRETE: DENTRO DO MODELO NÃO COLOCAR NUMERO NO NOME DAS VARIAVEIS

  X = x[0]
  S = x[1]
  P = x[2]
  V = x[3]

  #Declaração das equações constitutivas

  u = umax*(S/(Ks+S)) #TAXA DE CRESCIMENTO ESPECÍFICO
  rg = u*X #TAXA DE CRESCIMENTO DE CÉLULAS
  rp = Ypx*rg #TAXA DE FORMAÇÃO DE PRODUTO

  #Declaração das equações do modelo

  dV = F
  dX = ((1/V)*((V*rg)-(dV*X)))
  dP = ((1/V)*((V*rp)-(dV*P)))
  dS = ((1/V)*((F*Sf)-((1/Yxs)*V*rg)-(dV*S)))

  #Saida do modelo

  dx = [dX, dS, dP, dV]
  return dx

#Definição a função de medida para o cálculo

def medida(t,x,u,params={}):
    '''Função de medida'''

    # variáveis de estado
    X, S, P, V = x

    # modelo de observação
    y = [X, S, P, V]

    return y

# Instanciar o sistema I/O
reator = ct.NonlinearIOSystem(
            reator_batelada,
            outfcn = medida,
            states = ('X','S', 'P', 'V'),
            inputs = ('Sf','F'),
            outputs =('X','S', 'P', 'V'))

# condição inicial (3 pontos estacionários)

xs = [0.05, 10.0, 0.0, 1.0] # cA0, T0
us = [10.0, 0.02] # q0, cAi0, Tc0

reator_linear = linearize(reator, xeq=xs, ueq=us)

# modelo linear
A = reator_linear.A
B = reator_linear.B
C = reator_linear.C
D = reator_linear.D

sys = ct.ss(A,B,C,D)
print(sys)

# checar a estabilidade

p, v = linalg.eig(A)

print('\n autovalores = \n', p)
print('\n autovetores = \n', v)

if any(np.real(p)>0):
    print('\n O sistema é INSTÁVEL.')
else:
    print('\n O sistema é ESTÁVEL.')

<StateSpace>: sys[4]
Inputs (2): ['u[0]', 'u[1]']
Outputs (4): ['y[0]', 'y[1]', 'y[2]', 'y[3]']
States (4): ['x[0]', 'x[1]', 'x[2]', 'x[3]']

A = [[ 1.61818182e-01  8.26446203e-05  0.00000000e+00  9.99998997e-04]
     [-3.63636364e-01 -2.01652892e-02  0.00000000e+00  1.73472348e-11]
     [ 3.63636364e-02  1.65289241e-05 -2.00000000e-02  0.00000000e+00]
     [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]

B = [[ 0.   -0.05]
     [ 0.02  0.  ]
     [ 0.    0.  ]
     [ 0.    1.  ]]

C = [[1. 0. 0. 0.]
     [0. 1. 0. 0.]
     [0. 0. 1. 0.]
     [0. 0. 0. 1.]]

D = [[0. 0.]
     [0. 0.]
     [0. 0.]
     [0. 0.]]

 autovalores = 
 [-0.02      +0.j -0.02      +0.j  0.16165289+0.j  0.        +0.j]

 autovetores = 
 [[ 0.00000000e+00  4.54545363e-04 -4.45435403e-01 -6.19762497e-03]
 [ 0.00000000e+00 -9.99999892e-01  8.90870806e-01  1.11760451e-01]
 [ 1.00000000e+00  9.33204578e-05 -8.90870806e-02 -1.11760450e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  9.9365300